# RISC-V Pipelined Processor - PYNQ-Z2 Verification

| Block | AXI Base | Purpose |
|---|---|---|
| `bram_controller_iram` | `0x42000000` | Instruction BRAM (4096 x 32-bit) |
| `bram_controller_dram` | `0x40000000` | Data BRAM (4096 x 32-bit) |
| `axi_gpio_reset` | `0x41200000` | 1-bit GPIO -> core resetn |

GPIO=0 -> core halted, GPIO=1 -> core running

In [1]:
# =============================================================
# Cell 1: Load Overlay & Map Hardware
# =============================================================
from pynq import Overlay, MMIO
import time
import numpy as np

BITSTREAM = "riscv_pynq_lfg.bit"  # <-- YOUR BITSTREAM NAME

print("Loading Overlay...")
overlay = Overlay(BITSTREAM)
print("Overlay Loaded.")

print("IPs:", list(overlay.ip_dict.keys()))
print("Mem:", list(overlay.mem_dict.keys()))

# BRAM controllers are in mem_dict
iram_info = overlay.mem_dict['bram_controller_iram']
dram_info = overlay.mem_dict['bram_controller_dram']
iram = MMIO(iram_info['phys_addr'], iram_info['addr_range'])
dram = MMIO(dram_info['phys_addr'], dram_info['addr_range'])

# GPIO is in ip_dict
gpio_info = overlay.ip_dict['axi_gpio_reset']
gpio = MMIO(gpio_info['phys_addr'], gpio_info['addr_range'])

def halt_core():
    gpio.write(0x0, 0x0)

def run_core():
    gpio.write(0x0, 0x1)

print(f"IRAM: {iram_info['addr_range']//4} words, DRAM: {dram_info['addr_range']//4} words")
print("Hardware mapped.")

Loading Overlay...


Overlay Loaded.
IPs: ['axi_gpio_reset', 'processing_system7_0']
Mem: ['bram_controller_dram', 'bram_controller_iram', 'PSDDR']
IRAM: 4096 words, DRAM: 4096 words
Hardware mapped.


In [2]:
# =============================================================
# Cell 2: Sanity Check
# =============================================================
print("=" * 50)
print("SANITY CHECK")
print("=" * 50)
halt_core()
errors = 0

for pattern in [0xDEADBEEF, 0x01020304, 0xFFFFFFFF, 0x00000000]:
    iram.write(0x0, pattern)
    rb = iram.read(0x0)
    ok = rb == pattern
    if not ok: errors += 1
    print(f"  IRAM 0x{pattern:08X} -> 0x{rb:08X} [{'OK' if ok else 'FAIL'}]")

for pattern in [0xCAFEBABE, 0x12345678, 0x00000000]:
    dram.write(0x0, pattern)
    rb = dram.read(0x0)
    ok = rb == pattern
    if not ok: errors += 1
    print(f"  DRAM 0x{pattern:08X} -> 0x{rb:08X} [{'OK' if ok else 'FAIL'}]")

# Test multiple offsets
for off in [0x0, 0x4, 0x10, 0x7C, 0x100]:
    dram.write(off, 0xAA000000 | off)
    rb = dram.read(off)
    ok = rb == (0xAA000000 | off)
    if not ok: errors += 1
    print(f"  DRAM[0x{off:03X}] = 0x{rb:08X} [{'OK' if ok else 'FAIL'}]")

gpio.write(0x0, 0x0)
v0 = gpio.read(0x0) & 0x1
gpio.write(0x0, 0x1)
v1 = gpio.read(0x0) & 0x1
gpio_ok = (v0 == 0 and v1 == 1)
if not gpio_ok: errors += 1
print(f"  GPIO: 0->{v0}, 1->{v1} [{'OK' if gpio_ok else 'FAIL'}]")
halt_core()

print("=" * 50)
print("ALL PASSED" if errors == 0 else f"FAILED: {errors} error(s)")
print("=" * 50)

SANITY CHECK
  IRAM 0xDEADBEEF -> 0xDEADBEEF [OK]
  IRAM 0x01020304 -> 0x01020304 [OK]
  IRAM 0xFFFFFFFF -> 0xFFFFFFFF [OK]
  IRAM 0x00000000 -> 0x00000000 [OK]
  DRAM 0xCAFEBABE -> 0xCAFEBABE [OK]
  DRAM 0x12345678 -> 0x12345678 [OK]
  DRAM 0x00000000 -> 0x00000000 [OK]
  DRAM[0x000] = 0xAA000000 [OK]
  DRAM[0x004] = 0xAA000004 [OK]
  DRAM[0x010] = 0xAA000010 [OK]
  DRAM[0x07C] = 0xAA00007C [OK]
  DRAM[0x100] = 0xAA000100 [OK]
  GPIO: 0->0, 1->1 [OK]
ALL PASSED


In [3]:
# =============================================================
# Cell 3: Smoke Test (no hex file needed)
# =============================================================
print("=" * 50)
print("SMOKE TEST")
print("=" * 50)
halt_core()

# Clear memories
for i in range(16):
    iram.write(i * 4, 0x00000013)  # NOP
    dram.write(i * 4, 0xFFFFFFFF)  # sentinel

# addi x1,x0,42; sw x1,0(x0); addi x2,x0,7; sw x2,4(x0); beq x0,x0,0
smoke = [0x02A00093, 0x00102023, 0x00700113, 0x00202223, 0x00000063]
for i, instr in enumerate(smoke):
    iram.write(i * 4, instr)

run_core()
time.sleep(0.5)
halt_core()

v0 = dram.read(0x0)
v1 = dram.read(0x4)
print(f"DRAM[0] = {v0} (expect 42), DRAM[1] = {v1} (expect 7)")
if v0 == 42 and v1 == 7:
    print(">>> SMOKE TEST PASSED <<<")
else:
    print(">>> SMOKE TEST FAILED <<<")

SMOKE TEST
DRAM[0] = 42 (expect 42), DRAM[1] = 7 (expect 7)
>>> SMOKE TEST PASSED <<<


In [5]:
# =============================================================
# Cell 4: 32-Element Bubble Sort (EMBEDDED program - no file needed)
# =============================================================
# This is the simulation-verified 27-instruction loop-based bubble sort.
# It passed all 7 tests in simulation including full 32-element sort.
#
# Register usage:
#   x1  = i (outer loop counter, 0..N-2)
#   x2  = N-1 = 31
#   x3  = inner loop limit = N-1-i
#   x5  = j (inner loop counter)
#   x6  = byte offset = j*4
#   x7  = arr[j]
#   x8  = arr[j+1]
#   x9  = slt result
#   x10 = base address (always 0)
#
# Data layout: DRAM[0..31] = 32 integers, DRAM[64] (0x100) = done flag

PROGRAM = [
    0x00000093,  #  0: addi x1, x0, 0       ; i = 0
    0x01F00113,  #  1: addi x2, x0, 31      ; N-1 = 31
    0x00400513,  #  2: addi x10, x0, 4      ; stride = 4 (byte offset per word)
    0x00000193,  #  3: addi x3, x0, 0       ; (outer loop start)
    0x0021A4B3,  #  4: slt  x9, x3, x2      ; x9 = (i < N-1) ? -- OUTER LOOP
    0x04048263,  #  5: beq  x9, x0, +68     ; if i >= N-1, jump to DONE (PC+68 -> instr 22)
    0x40310233,  #  6: sub  x4, x2, x3      ; x4 = N-1-i (inner limit)
    0x00000293,  #  7: addi x5, x0, 0       ; j = 0
    0x0042A4B3,  #  8: slt  x9, x5, x4      ; x9 = (j < N-1-i) ? -- INNER LOOP
    0x02048663,  #  9: beq  x9, x0, +44     ; if j >= limit, jump to outer incr (PC+44 -> instr 20)
    0x00229313,  # 10: slli x6, x5, 2       ; x6 = j * 4
    0x00608333,  # 11: add  x6, x1, x6      ; x6 = base + j*4 (but base=0, so x6=j*4) -- NOTE: x1 is 'i' but also 0-based addr. Actually x1=i, we use x6=j*4 directly
    0x00032383,  # 12: lw   x7, 0(x6)       ; x7 = arr[j]
    0x00432403,  # 13: lw   x8, 4(x6)       ; x8 = arr[j+1]
    0x007424B3,  # 14: slt  x9, x8, x7      ; x9 = (arr[j+1] < arr[j]) ?
    0x00048663,  # 15: beq  x9, x0, +12     ; if not, skip swap (PC+12 -> instr 18)
    0x00832023,  # 16: sw   x8, 0(x6)       ; arr[j] = arr[j+1]
    0x00732223,  # 17: sw   x7, 4(x6)       ; arr[j+1] = arr[j]
    0x00128293,  # 18: addi x5, x5, 1       ; j++
    0xFC000AE3,  # 19: beq  x0, x0, -44     ; jump back to inner loop (PC-44 -> instr 8) -- CORRECTED: should be -52? Let me check
    0x00118193,  # 20: addi x3, x3, 1       ; i++
    0xFA000EE3,  # 21: beq  x0, x0, -68     ; jump back to outer loop (PC-68 -> instr 4) -- CORRECTED
    0xDEADC637,  # 22: lui  x12, 0xDEADC    ; DONE: load upper bits of 0xDEADBEAF
    0xEAF60613,  # 23: addi x12, x12, -337  ; x12 = 0xDEADBEAF
    0x10000693,  # 24: addi x13, x0, 256    ; x13 = 0x100 (status flag byte offset)
    0x00C6A023,  # 25: sw   x12, 0(x13)     ; DRAM[0x100] = 0xDEADBEAF
    0x00000063,  # 26: beq  x0, x0, 0       ; infinite loop (halt)
]

# NOTE on instruction 11: x1 starts as i=0 and increments each outer pass.
# But x6 = j*4 is the correct byte offset since base address is 0.
# Actually looking at this more carefully: x1 = i (outer counter), NOT base addr.
# Instruction 11 does: x6 = x1 + x6 = i + j*4. That's WRONG for addressing.
# But wait - this is the hex that passed all simulation tests. Let me re-check.
# 
# Actually x1 is initialized to 0 at instruction 0 and NEVER modified in the
# inner loop. x3 is the outer counter (i), not x1. So x1=0 always.
# Instruction 11: x6 = x1(=0) + x6(=j*4) = j*4. Correct!

STATUS_BYTE = 0x100   # byte offset for done flag (word 64)
DONE_FLAG = 0xDEADBEAF
TIMEOUT = 5

np.random.seed(42)

test_data = np.random.randint(-300, 300, size=32).tolist()
golden = sorted(test_data)

print("=" * 60)
print("BUBBLE SORT TEST (32 signed integers)")
print(f"Program: {len(PROGRAM)} instructions (embedded, no file needed)")
print("=" * 60)

# --- Step 1: Halt ---
halt_core()
print("[1] Core halted.")

# --- Step 2: Clear IRAM (fill with NOPs) then load program ---
# Clear first 64 words to avoid stale instructions
for i in range(64):
    iram.write(i * 4, 0x00000013)  # NOP
for i, instr in enumerate(PROGRAM):
    iram.write(i * 4, instr)
print(f"[2] Loaded {len(PROGRAM)} instructions into IRAM.")

# Verify all instructions
iram_ok = True
for i, instr in enumerate(PROGRAM):
    v = iram.read(i * 4)
    if v != instr:
        print(f"    IRAM[{i}] MISMATCH: wrote 0x{instr:08X}, read 0x{v:08X}")
        iram_ok = False
if iram_ok:
    print(f"    All {len(PROGRAM)} instructions verified OK.")
else:
    print("    !!! IRAM VERIFICATION FAILED - DO NOT PROCEED !!!")

# --- Step 3: Load test data into DRAM ---
for i, val in enumerate(test_data):
    dram.write(i * 4, val & 0xFFFFFFFF)
dram.write(STATUS_BYTE, 0x00000000)  # clear done flag
print(f"[3] Loaded {len(test_data)} data words + cleared status flag.")

# Verify data was written correctly
data_ok = True
for i, val in enumerate(test_data):
    v = dram.read(i * 4)
    expected = val & 0xFFFFFFFF
    if v != expected:
        sv = v if v < 0x80000000 else v - 0x100000000
        print(f"    DRAM[{i}] MISMATCH: wrote {val}, read {sv} (0x{v:08X})")
        data_ok = False
if data_ok:
    print(f"    All {len(test_data)} data words verified OK.")
else:
    print("    !!! DATA VERIFICATION FAILED - DO NOT PROCEED !!!")

# --- Step 4: Run ---
print("[4] Releasing reset...")
run_core()

# --- Step 5: Poll for completion ---
start = time.time()
done = False
while (time.time() - start) < TIMEOUT:
    flag = dram.read(STATUS_BYTE)
    if flag == DONE_FLAG:
        done = True
        break
    time.sleep(0.001)
elapsed = time.time() - start
halt_core()
print(f"[5] Flag: 0x{dram.read(STATUS_BYTE):08X} ({'DONE' if done else 'NOT SET'}) after {elapsed:.4f}s")

# --- Step 6: Read & verify ---
print()
result = []
for i in range(32):
    v = dram.read(i * 4)
    sv = v if v < 0x80000000 else v - 0x100000000
    result.append(sv)

print("Output:  ", result)
print("Expected:", golden)

errors = 0
for i in range(32):
    if result[i] != golden[i]:
        print(f"  MISMATCH [{i}]: got {result[i]}, expected {golden[i]}")
        errors += 1

print()
print("=" * 60)
if done and errors == 0:
    print("  >>> ALL 32 ELEMENTS SORTED CORRECTLY <<<")
elif not done:
    print("  FAIL: Core did not write completion flag.")
    print("  Check: Is the bitstream built with the NEW IP (instruction buffer fix)?")
else:
    print(f"  FAIL: {errors} element(s) wrong.")
print("=" * 60)

BUBBLE SORT TEST (32 signed integers)
Program: 27 instructions (embedded, no file needed)
[1] Core halted.
[2] Loaded 27 instructions into IRAM.
    All 27 instructions verified OK.
[3] Loaded 32 data words + cleared status flag.
    All 32 data words verified OK.
[4] Releasing reset...
[5] Flag: 0xDEADBEAF (DONE) after 0.0016s

Output:   [-280, -279, -242, -229, -213, -201, -198, -194, -179, -170, -140, -109, -86, -48, -30, -24, 8, 13, 30, 43, 72, 85, 113, 135, 158, 159, 166, 174, 175, 191, 210, 260]
Expected: [-280, -279, -242, -229, -213, -201, -198, -194, -179, -170, -140, -109, -86, -48, -30, -24, 8, 13, 30, 43, 72, 85, 113, 135, 158, 159, 166, 174, 175, 191, 210, 260]

  >>> ALL 32 ELEMENTS SORTED CORRECTLY <<<


In [ ]:
# =============================================================
# Cell 5: Debug Dump (run if sort fails)
# =============================================================
halt_core()

print("--- IRAM (first 30 words) ---")
for i in range(30):
    v = iram.read(i * 4)
    print(f"  [{i:2d}] 0x{i*4:03X}: 0x{v:08X}")

print("\n--- DRAM (first 40 words) ---")
for i in range(40):
    v = dram.read(i * 4)
    sv = v if v < 0x80000000 else v - 0x100000000
    print(f"  [{i:2d}] 0x{i*4:03X}: 0x{v:08X} ({sv})")

print(f"\n--- Status flag (0x100) ---")
print(f"  0x{dram.read(0x100):08X}")